# ![](https://ga-dash.s3.amazonaws.com/production/assets/logo-9f88ae6c9c3871690e33280fcf557f33.png) Lesson 3.04 | Regression Metrics

### Introduction

We've learned a lot about linear regression models thus far.
- How to fit linear regression models.
- How to identify candidates for independent variables.
- The assumptions of linear regression models and how to assess their validity.

One thing we haven't discussed in significant detail is how to measure the performance of a regression model. 
- What makes a good model?
- What makes a bad model?
- Between two good models, which should we pick and why?

**Discussion:** What makes a good model versus a bad model? How can we measure that?

Metrics, or measurements, will allow us to directly compare models that attempt to achieve the same goal.
- Note: Depending on the metric, we might need to be wary of the scale of our $Y$ values!

**The Process of Data Science**
1. Data Gathering
2. Data Cleaning/Munging
3. EDA
4. Modeling
5. Reporting
    - Model evaluation will mostly be used in steps 4 and 5.

Once we build models, we want to [evaluate their performance](http://scikit-learn.org/stable/modules/model_evaluation.html)!

### Regression Metrics
We're going to go through six different regression metrics.
1. Mean Squared Error
2. Root Mean Squared Error
3. Mean Squared Log Error
4. Median Absolute Error
5. $R^2$
6. Adjusted $R^2$

In [1]:
import sklearn.metrics as metrics
import sklearn.linear_model as linear_model
import pandas as pd

In [2]:
df = pd.read_csv("forestfires.csv")

In [3]:
dep = df['area']
indep = df.drop(['month','day','area'], axis = 'columns')

In [4]:
lm = linear_model.LinearRegression(fit_intercept=True,
                                   normalize=False,
                                   copy_X=True,
                                   n_jobs=1)

In [5]:
lm.fit(X=indep, y=dep)

LinearRegression(copy_X=True, fit_intercept=True, n_jobs=1, normalize=False)

In [6]:
lm.coef_

array([  1.90794524e+00,   5.69180991e-01,  -3.91995160e-02,
         7.73345196e-02,  -3.29461471e-03,  -7.13739029e-01,
         8.00212731e-01,  -2.30644773e-01,   1.55743147e+00,
        -3.40403700e+00])

In [7]:
lm.intercept_

-6.3693147685743501

In [8]:
predicted = lm.predict(X=indep) # y_preds

In [9]:
true = dep # y_true

#### Mean Squared Error

$$MSE(\mathbf{y},\hat{\mathbf{y}}) = \frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2$$

##### ![](https://upload.wikimedia.org/wikipedia/commons/e/ed/Residuals_for_Linear_Regression_Fit.png)

Goal: Get $MSE$ as close to 0 as possible.

Pros: 
- Very common; part of other calculations.
- Represents average distance squared from the predicted value.

Cons: 
- Can be heavily affected by outliers.
- Not in the original units of $Y$.
- Depends on scale of $Y$. (i.e. housing prices vs. GPA)

[Documentation](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_error.html#sklearn.metrics.mean_squared_error)

In [10]:
metrics.mean_squared_error(y_pred = predicted,
                           y_true = true)

3956.7092182333949

#### Root Mean Squared Error

$$RMSE(\mathbf{y},\hat{\mathbf{y}}) = \sqrt{\frac{1}{n}\sum_{i=1}^n(y_i-\hat{y}_i)^2}$$

Goal: Get $RMSE$ as close to 0 as possible.

Pros: 
- Pretty common; part of other calculations.
- Represents (approximately) average distance from the predicted value.
- In the original units of $Y$.

Cons: 
- Can be heavily affected by outliers.
- Depends on scale of $Y$. (i.e. housing prices vs. GPA)

There's no documentation, but it should be quite easy to generate the value.

In [11]:
metrics.mean_squared_error(y_pred = predicted,
                           y_true = true) ** 0.5

62.902378478348453

#### Mean Squared Log Error
$$MSLE = \frac{1}{n}\sum_{i=1}^n\left(\log(\hat{y} + 1) - log(y + 1)\right) ^ 2$$

Note: $\log(\cdot)$ in this case refers to the natural logarithm or the logarithm with base $e \approx 2.71828$. 

Goal: Get $MSLE$ as close to 0 as possible.

Pros: 
- If predicted and observed values are both large, their difference is likely going to be large as well. The logarithm helps to account for that scale.

Cons: 
- Not in the original values of $Y$.
- Depends on scale of $Y$. (i.e. housing prices vs. GPA)
- $\log(0)$ is not defined.

[Documentation](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_squared_log_error.html#sklearn.metrics.mean_squared_log_error)

In [12]:
metrics.mean_squared_log_error(y_pred = predicted,
                               y_true = true)

/Users/mattbrems/anaconda3/lib/python3.5/site-packages/sklearn/metrics/regression.py:314: RuntimeWarning: invalid value encountered in log
  return mean_squared_error(np.log(y_true + 1), np.log(y_pred + 1),


ValueError: Input contains NaN, infinity or a value too large for dtype('float64').

#### Median Absolute Error

$$MedAE(\mathbf{y},\hat{\mathbf{y}}) = \text{median}\left(\left|y_1-\hat{y}_1\right|,\ldots ,\left|y_n-\hat{y}_n\right|\right)$$

Goal: Get $MedAE$ as close to 0 as possible.

Pros: 
- Represents median distance from the predicted value.
- In the original units of $Y$. 
- Is not heavily affected by outliers.

Cons: 
- Depends on scale of $Y$. (i.e. housing prices vs. GPA)

[Documentation](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.mean_absolute_error.html#sklearn.metrics.mean_absolute_error)

In [13]:
metrics.median_absolute_error(y_pred = predicted,
                              y_true = true)

12.166843506305414

In [14]:
metrics.mean_absolute_error(y_pred = predicted,
                            y_true = true)

19.296884540072281

#### Coefficient of Determination, $R^2$

$$R^2 = 1 - \frac{\sum_{i=1}^n(y_i - \hat{y}_i)^2}{\sum_{i=1}^n(y_i - \bar{y})^2}$$

Goal: Get $R^2$ as close to 1 as possible.

Pros:
- Easy interpretation. "An $R^2$ value of 0.8 means that 80% of the variability in the data are explained by our model, relative to a model with no predictors."
- Common metric.
- Does not depend on the scale of $Y$.

Cons:
- As you add more variables, $R^2$ will never decrease.

[Documentation](http://scikit-learn.org/stable/modules/generated/sklearn.metrics.r2_score.html#sklearn.metrics.r2_score)

In [15]:
metrics.r2_score(y_pred = predicted,
                 y_true = true)

0.021639831514046137

#### Adjusted $R^2$

$$R^2_{adj} = 1 - \frac{\frac{1}{n-p-1}\sum_{i=1}^n(y_i - \hat{y}_i)^2}{\frac{1}{n-1}\sum_{i=1}^n(y_i - \bar{y})^2}$$

Goal: Get $R^2_{adj}$ as close to 1 as possible.

Pros:
- Does not depend on the scale of $Y$.
- Penalizes for "bad" variables.

Cons:
- Interpretation isn't exactly the same, although we often treat it as such.

### Feature Selection

Slightly different from feature engineering is **feature selection**. Feature selection is (unsurprisingly) the process by which we select which features to include in our model.

- Suppose we have five features. How might we select which ones to include when modeling?
- Suppose we have five hundred features. How might we select which ones to include when modeling?

It is to our benefit to have an [automated means](http://scikit-learn.org/stable/modules/feature_selection.html) of identifying features to include/exclude.

#### Forward Selection
1. Start with null model (just $\hat{y} = \beta_0$).
2. Add one feature by testing all possible features $X_i$ and picking the **best** based on some metric.
3. Repeat until a stopping condition is met.

#### Backward Selection
1. Start with full model ($\hat{y} = \beta_0 + \beta_1X_1 + \cdots \beta_pX_p$).
2. Remove one feature by testing all possible features $X_i$ and picking the **worst** based on some metric.
3. Repeat step 2 until a stopping condition is met.

**NOTE: READ BEFORE TOMORROW! http://scott.fortmann-roe.com/docs/BiasVariance.html**